# How this notebook is laid out
Refer to Section 3.6: Building a Knowledge-Based Agent
https://www.ricopic.one/teaching/notes/engineering-artificial-intelligence/knowledge-and-reasoning/building-a-knowledge-based-agent/#k7


In [1]:
# Verify required packages are available
import sys
print(f"Python version: {sys.version}")

try:
    import numpy as np
    print(f"✓ numpy {np.__version__} is available")
except ImportError as e:
    print(f"✗ numpy not available: {e}")

try:
    import matplotlib.pyplot as plt
    print("✓ matplotlib is available")
except ImportError as e:
    print(f"✗ matplotlib not available: {e}")

print("All required packages are ready!")

Python version: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.6.3.2)]
✓ numpy 2.4.1 is available
✓ matplotlib is available
All required packages are ready!


# What is Z3?
Z3 is an open-source SMT (Satisfiability Modulo Theories) solver developed by Microsoft Research. It is an industrial-strength tool used in software verification, security analysis, and AI research. It supports:

Propositional logic with native biconditionals (no manual CNF conversion)
First-order logic with quantifiers (, ) over user-defined sorts
Arithmetic, bit-vectors, arrays, and other theories
We will use Z3 as our reasoning engine throughout this chapter. In this section we use its propositional logic capabilities; in Section 3.7: Building a FOL Agent with Z3 we extend to first-order logic.

In [8]:
# Check if z3-solver is already installed
try:
    import z3
    print(f"✓ z3-solver is already available! Version info: {z3.get_version_string()}")
except ImportError:
    print("✗ z3-solver not available - trying installation with break-system-packages...")
    
    import subprocess
    import sys
    
    # Try with --break-system-packages and --user for safety
    try:
        print("Installing z3-solver with --break-system-packages --user...")
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", "--user", "z3-solver"], 
                              capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            print("✓ z3-solver installed successfully!")
            print("Refreshing imports...")
            
            # Invalidate import caches and try importing
            import importlib
            importlib.invalidate_caches()
            
            # Restart the kernel might be needed, but let's try importing first
            try:
                import z3
                print(f"✓ z3-solver imported successfully! Version: {z3.get_version_string()}")
            except ImportError:
                print("⚠️  z3-solver installed but import failed - kernel restart may be needed")
                print("Try restarting the kernel and running this cell again")
        else:
            print(f"✗ Installation failed: {result.stderr}")
            print("\nAlternative solutions:")
            print("1. Install via terminal: python3 -m pip install --break-system-packages --user z3-solver")
            print("2. Use conda if available: conda install z3-solver")
            print("3. Try: brew install z3-solver (if using Homebrew)")
    except subprocess.TimeoutExpired:
        print("✗ Installation timed out (>5 minutes)")
        print("z3-solver can take a long time to compile on some systems")
    except Exception as e:
        print(f"✗ Unexpected error: {e}")

✓ z3-solver is already available! Version info: 4.15.8


In [2]:
# Quick test to verify z3-solver is working correctly
import z3

# Create a simple SAT problem to test
x, y = z3.Ints('x y')
solver = z3.Solver()
solver.add(x + y == 10)
solver.add(x > 5)

if solver.check() == z3.sat:
    model = solver.model()
    print("✓ z3-solver is working correctly!")
    print(f"Example solution: x={model[x]}, y={model[y]}")
    print("You can now proceed with your theorem proving tasks!")
else:
    print("✗ z3-solver test failed")

✓ z3-solver is working correctly!
Example solution: x=6, y=4
You can now proceed with your theorem proving tasks!


In [3]:
# Another quick example to verify the installation

from z3 import Bool, Bools, Solver, And, Or, Not
P, Q = Bools('P Q')
s = Solver()
s.add(P == Q)    # Biconditional --- native, no CNF needed
s.add(P)
print(s.check())  # sat
print(s.model())   # [Q = True, P = True]

sat
[Q = True, P = True]


In [2]:
# Sets up the warehouse environment
from dataclasses import dataclass
import random
from typing import Dict, List, Tuple, Union
Action = Union[int, str]
@dataclass
class WarehouseState:
    robot_pos: Tuple[int, int]
    has_item: bool
    battery: int
    steps: int
class WarehouseEnv:
    """
    Minimal, Gymnasium-style warehouse environment.
    - reset() -> observation
    - step(action) -> observation, reward, terminated, truncated, info
    """
    # Discrete action set for the agent.
    ACTIONS = ["N", "E", "S", "W", "WAIT", "PICK", "DROP"]
    MOVE_DELTAS = {
        "N": (-1, 0),
        "E": (0, 1),
        "S": (1, 0),
        "W": (0, -1),
    }
    def __init__(
        self,
        grid: List[str] | None = None,
        start_pos: Tuple[int, int] = (1, 1),
        max_steps: int = 200,
        battery: int = 200,
        view_radius: int = 2,
    ) -> None:
        # Legend: # = wall, . = empty, P = pickup, D = dropoff.
        self.grid = grid or [
            "############",
            "#..P....#..#",
            "#..##...#..#",
            "#......##..#",
            "#..#.......#",
            "#..#..D....#",
            "############",
        ] # Default warehouse layout
        self.height = len(self.grid)
        self.width = len(self.grid[0])
        self.start_pos = start_pos
        self.max_steps = max_steps
        self.max_battery = battery
        self.view_radius = view_radius
        self.state = WarehouseState(
            robot_pos=self.start_pos,
            has_item=False,
            battery=self.max_battery,
            steps=0,
        )
    def reset(self, randomize: bool = False) -> Dict[str, object]:
        start_pos = self.start_pos
        if randomize:
            self._randomize_pickup_dropoff()
            start_pos = self._random_empty_cell()
        self.state = WarehouseState(
            robot_pos=start_pos,
            has_item=False,
            battery=self.max_battery,
            steps=0,
        )
        return self._observe()
    def step(self, action: Action) -> Tuple[Dict[str, object], float, bool, bool, Dict[str, object]]:
        act = self._normalize_action(action)
        # Small step penalty encourages shorter paths.
        reward = -0.1
        terminated = False
        truncated = False
        info: Dict[str, object] = {}
        if act in self.MOVE_DELTAS:
            reward += self._move(act)
        elif act == "WAIT":
            reward -= 0.05
        elif act == "PICK":
            reward += self._pick()
        elif act == "DROP":
            reward += self._drop()
            if reward >= 9.0:
                terminated = True
        else:
            reward -= 0.5
            info["invalid_action"] = True
        self.state.steps += 1
        self.state.battery -= 1
        # Truncate when the time or battery budget runs out.
        if self.state.steps >= self.max_steps or self.state.battery <= 0:
            truncated = True
        return self._observe(), reward, terminated, truncated, info
    def render_grid(self) -> List[List[str]]:
        """Return a 2D grid of characters for animation or visualization."""
        rows = [list(r) for r in self.grid]
        r, c = self.state.robot_pos
        rows[r][c] = "R" if not self.state.has_item else "r"
        return rows
    def render(self) -> str:
        # Render the full grid with the robot position overlaid.
        rows = self.render_grid()
        return "\n".join("".join(r) for r in rows)
    def render_with_legend(self) -> str:
        legend = [
            "Legend:",
            "# = wall",
            ". = empty",
            "P = pickup",
            "D = dropoff",
            "R = robot (empty)",
            "r = robot (loaded)",
        ]
        return f"{self.render()}\n\n" + "\n".join(legend)
    def _normalize_action(self, action: Action) -> str:
        if isinstance(action, int):
            if 0 <= action < len(self.ACTIONS):
                return self.ACTIONS[action]
            return "INVALID"
        return action.upper()
    def _move(self, act: str) -> float:
        dr, dc = self.MOVE_DELTAS[act]
        r, c = self.state.robot_pos
        nr, nc = r + dr, c + dc
        if self._is_wall(nr, nc):
            return -1.0
        self.state.robot_pos = (nr, nc)
        return 0.0
    def _pick(self) -> float:
        r, c = self.state.robot_pos
        # Successful pickup is only allowed on a pickup tile.
        if self.grid[r][c] == "P" and not self.state.has_item:
            self.state.has_item = True
            return 5.0
        return -0.5
    def _drop(self) -> float:
        r, c = self.state.robot_pos
        # Successful drop is only allowed on a dropoff tile.
        if self.grid[r][c] == "D" and self.state.has_item:
            self.state.has_item = False
            return 10.0
        return -0.5
    def _is_wall(self, r: int, c: int) -> bool:
        if r < 0 or c < 0 or r >= self.height or c >= self.width:
            return True
        return self.grid[r][c] == "#"
    def _observe(self) -> Dict[str, object]:
        # Local observation centered on the robot, using view_radius.
        r, c = self.state.robot_pos
        local = []
        for dr in range(-self.view_radius, self.view_radius + 1):
            row = []
            for dc in range(-self.view_radius, self.view_radius + 1):
                rr, cc = r + dr, c + dc
                if rr < 0 or cc < 0 or rr >= self.height or cc >= self.width:
                    row.append("#")
                elif (rr, cc) == self.state.robot_pos:
                    row.append("R" if not self.state.has_item else "r")
                else:
                    row.append(self.grid[rr][cc])
            local.append("".join(row))
        pickup_pos = self._find_tile("P")
        dropoff_pos = self._find_tile("D")
        return {
            "local_grid": local,
            "robot_pos": self.state.robot_pos,
            "has_item": self.state.has_item,
            "battery": self.state.battery,
            "steps": self.state.steps,
            "pickup_pos": pickup_pos,
            "dropoff_pos": dropoff_pos,
        }
    def _random_empty_cell(self) -> Tuple[int, int]:
        empties = []
        for r, row in enumerate(self.grid):
            for c, ch in enumerate(row):
                if ch == ".":
                    empties.append((r, c))
        if not empties:
            return self.start_pos
        return random.choice(empties)
    def _randomize_pickup_dropoff(self) -> None:
        # Convert to mutable grid.
        rows = [list(r) for r in self.grid]
        positions = []
        for r, row in enumerate(rows):
            for c, ch in enumerate(row):
                if ch in {"P", "D"}:
                    rows[r][c] = "."
                if ch == ".":
                    positions.append((r, c))
        if len(positions) < 2:
            self.grid = ["".join(r) for r in rows]
            return
        pickup = random.choice(positions)
        positions.remove(pickup)
        dropoff = random.choice(positions)
        pr, pc = pickup
        dr, dc = dropoff
        rows[pr][pc] = "P"
        rows[dr][dc] = "D"
        self.grid = ["".join(r) for r in rows]
    def _find_tile(self, tile: str) -> Tuple[int, int] | None:
        for r, row in enumerate(self.grid):
            for c, ch in enumerate(row):
                if ch == tile:
                    return (r, c)
        return None

# 3.7.1 From Propositiont to Predicates

## 3.7.1.1 Defining the Domain

In propositional logic, we created one Bool per square: Bool(f'D_{x}_{y}'). In FOL, we instead declare a Location sort and define predicates as Z3 Function objects:

In [1]:
from z3 import DeclareSort, Function, BoolSort, Const, ForAll, Exists, Distinct
Location = DeclareSort('Location')
Damaged_fn  = Function('Damaged',  Location, BoolSort())
Forklift_fn = Function('Forklift', Location, BoolSort())
Creaking_fn = Function('Creaking', Location, BoolSort())
Rumbling_fn = Function('Rumbling', Location, BoolSort())
Safe_fn     = Function('Safe',     Location, BoolSort())
Adjacent_fn = Function('Adjacent', Location, Location, BoolSort())

Each Function maps Location (or pairs of locations) to booleans—exactly the predicates from Section 3.5.5: Predicates: Properties and Relations in Section 3.5: First-Order Logic. We also create a constant for each grid square:

In [2]:
loc = {}
width = globals().get('width', 4)
height = globals().get('height', 4)
for x in range(1, width + 1):
    for y in range(1, height + 1):
        loc[(x, y)] = Const(f'L_{x}_{y}', Location)

Compare with the propositional approach from Section 3.4: Building a Knowledge-Based Agent, where damaged(3, 1) returned Bool('D_3_1'). Here, Damaged_fn(loc[(3, 1)]) applies the predicate Damaged to the location constant L_3_1—a fundamentally different representation.

## 3.7.1.2 Quantified Physics Rules
Now we can write the creaking rule as a single quantified sentence:

In [5]:
from z3 import Solver, And
solver = Solver()
L  = Const('L',  Location)
Lp = Const('Lp', Location)
solver.add(ForAll(L,
    Creaking_fn(L) == Exists(Lp, And(Adjacent_fn(L, Lp), Damaged_fn(Lp)))
))

This is a direct translation of equation (3.5) from Section 3.5: First-Order Logic:
![image.png](equation_1.png)
One sentence for all locations—no loop over grid squares. The rumbling and safety rules are equally concise:

In [9]:
from z3 import Not
solver.add(ForAll(L,
    Rumbling_fn(L) == Exists(Lp, And(Adjacent_fn(L, Lp), Forklift_fn(Lp)))
))
solver.add(ForAll(L,
    Safe_fn(L) == And(Not(Damaged_fn(L)), Not(Forklift_fn(L)))
))

Three sentences encode the entire physics of the warehouse, regardless of whether the grid is  or . Compare this with the propositional agent from Section 3.4: Building a Knowledge-Based Agent, which needed 48 grounded biconditionals for a  grid.

## 3.7.1.3 Structural Facts: Adjacency and Domain Closure
The quantified rules express general physics, but the solver also needs to know the structure of the grid: which locations exist and which pairs are adjacent. These structural facts do require enumeration—but they encode the topology, not the physics.

Adjacency is a closed-world assertion: every pair of grid squares is either adjacent or not.

In [12]:
from z3 import Not

def get_adjacent(x, y, width=4, height=4):
    neighbors = []
    if x > 1:
        neighbors.append((x - 1, y))
    if x < width:
        neighbors.append((x + 1, y))
    if y > 1:
        neighbors.append((x, y - 1))
    if y < height:
        neighbors.append((x, y + 1))
    return neighbors

for x in range(1, width + 1):
    for y in range(1, height + 1):
        adj_set = set(get_adjacent(x, y, width, height))
        for x2 in range(1, width + 1):
            for y2 in range(1, height + 1):
                if (x2, y2) in adj_set:
                    solver.add(Adjacent_fn(loc[(x, y)], loc[(x2, y2)]))
                else:
                    solver.add(Not(Adjacent_fn(loc[(x, y)], loc[(x2, y2)])))

Domain closure is a subtlety that arises because Z3's DeclareSort creates an uninterpreted sort—the solver is free to imagine additional elements beyond our grid constants. Without domain closure, ForAll(L, ...) ranges over these phantom locations too. This breaks process-of-elimination reasoning: if the agent hears creaking at , the solver should conclude that one of 's neighbors has damaged floor. But if phantom locations could also be adjacent to  and absorb the blame, the solver can construct models where no real grid square is damaged—and the entailment fails.

The fix is a single axiom stating that every Location is one of our grid constants:

In [14]:
from z3 import Or, Distinct
solver.add(ForAll(L,
    Or([L == loc[(x, y)]
        for x in range(1, width + 1)
        for y in range(1, height + 1)])
))
solver.add(Distinct(list(loc.values())))

With domain closure, ForAll and Exists range only over actual grid squares, and the FOL encoding produces exactly the same entailment results as the propositional version from Section 3.4: Building a Knowledge-Based Agent.

## 3.7.1.4 The Complete FOL KB Builder
Putting it all together, build_warehouse_kb_fol creates the solver, declares the domain, adds structural facts, and encodes the quantified rules:

In [15]:
def build_warehouse_kb_fol(width=4, height=4):
    Location = DeclareSort('Location')
    Damaged_fn  = Function('Damaged',  Location, BoolSort())
    Forklift_fn = Function('Forklift', Location, BoolSort())
    Creaking_fn = Function('Creaking', Location, BoolSort())
    Rumbling_fn = Function('Rumbling', Location, BoolSort())
    Safe_fn     = Function('Safe',     Location, BoolSort())
    Adjacent_fn = Function('Adjacent', Location, Location, BoolSort())
    loc = {}
    for x in range(1, width + 1):
        for y in range(1, height + 1):
            loc[(x, y)] = Const(f'L_{x}_{y}', Location)
    solver = Solver()
    # Domain closure
    L = Const('L', Location)
    solver.add(ForAll(L,
        Or([L == loc[(x, y)]
            for x in range(1, width + 1)
            for y in range(1, height + 1)])
    ))
    solver.add(Distinct(list(loc.values())))
    # Adjacency (closed-world)
    for x in range(1, width + 1):
        for y in range(1, height + 1):
            adj_set = set(get_adjacent(x, y, width, height))
            for x2 in range(1, width + 1):
                for y2 in range(1, height + 1):
                    if (x2, y2) in adj_set:
                        solver.add(Adjacent_fn(loc[(x, y)], loc[(x2, y2)]))
                    else:
                        solver.add(Not(Adjacent_fn(loc[(x, y)], loc[(x2, y2)])))
    # Quantified physics rules
    Lp = Const('Lp', Location)
    solver.add(ForAll(L,
        Creaking_fn(L) == Exists(Lp, And(Adjacent_fn(L, Lp), Damaged_fn(Lp)))
    ))
    solver.add(ForAll(L,
        Rumbling_fn(L) == Exists(Lp, And(Adjacent_fn(L, Lp), Forklift_fn(Lp)))
    ))
    solver.add(ForAll(L,
        Safe_fn(L) == And(Not(Damaged_fn(L)), Not(Forklift_fn(L)))
    ))
    # Initial knowledge
    solver.add(Safe_fn(loc[(1, 1)]))
    predicates = {
        'Creaking': Creaking_fn, 'Rumbling': Rumbling_fn,
        'Safe': Safe_fn, 'Damaged': Damaged_fn,
        'Forklift': Forklift_fn, 'Adjacent': Adjacent_fn,
    }
    return solver, loc, predicates

The function returns a (solver, loc, predicates) tuple. The agent stores loc (to map grid coordinates to Z3 location constants) and predicates (to construct TELL and ASK expressions).

# 3.7.2 The Z3 FOL Agent
The WarehouseZ3Agent class mirrors the WarehouseKBAgent from Section 3.4: Building a Knowledge-Based Agent in decision strategy, path planning, and action conversion. The difference is the reasoning engine: Z3 with quantified FOL rules replaces the propositional grounded encoding.

## 3.7.2.1 Initialization
The agent builds the FOL KB and stores the location map and predicates:

In [16]:
def __init__(self, env):
    self.env = env
    self.solver, self.loc, self.preds = build_warehouse_kb_fol(
        env.width, env.height
    )
    # ... position, direction, visited, known_safe, etc.

## 3.7.2.2 TELL: Percepts as FOL Assertions
When the agent perceives creaking or rumbling, it TELLs the solver by applying the predicate function to the current location constant:

In [17]:
def tell_percepts(self, percept):
    L = self.loc[(self.x, self.y)]
    if percept.creaking:
        self.solver.add(self.preds['Creaking'](L))
    else:
        self.solver.add(Not(self.preds['Creaking'](L)))
    if percept.rumbling:
        self.solver.add(self.preds['Rumbling'](L))
    else:
        self.solver.add(Not(self.preds['Rumbling'](L)))

Compare with the propositional agent's solver.add(creaking_at(x, y)) from Section 3.4: Building a Knowledge-Based Agent. Here we write self.preds['Creaking'](L) — applying a FOL predicate to a location constant, rather than looking up a ground propositional symbol.

## 3.7.2.3 ASK: Safety Queries
The agent queries safety using z3_entails with the Safe predicate:

In [18]:
def update_safety(self):
    for x in range(1, self.env.width + 1):
        for y in range(1, self.env.height + 1):
            pos = (x, y)
            if pos in self.known_safe or pos in self.known_dangerous:
                continue
            L = self.loc[pos]
            if z3_entails(self.solver, self.preds['Safe'](L)):
                self.known_safe.add(pos)
            elif z3_entails(self.solver, Not(self.preds['Safe'](L))):
                self.known_dangerous.add(pos)

The rest of the agent—plan_path, path_to_actions, choose_action, execute_action, run—is identical to Section 3.4: Building a Knowledge-Based Agent.

# 3.7.3 Complete Implementation
The following module contains the complete FOL agent. It includes the quantified FOL encoding used by the agent and the full agent class.

In [3]:
"""
FOL Agent for the Hazardous Warehouse (Z3 Version)
Uses Z3's SMT solver with quantified first-order logic to reason about
safety and navigate the warehouse to retrieve the package.
This agent implements the same TELL/ASK loop as warehouse_kb_agent.py
but replaces the DPLL-based PropKB with Z3's Solver and expresses the
physics rules as quantified FOL sentences:
  1. TELL the solver about percepts (solver.add)
  2. ASK via entailment check (push/Not(query)/check/pop)
  3. Plan a path through safe squares toward the goal
  4. Execute actions and repeat
The physics rules are expressed as single universally quantified sentences:
  - ForAll L, Creaking(L) <=> Exists L', Adjacent(L,L') & Damaged(L')
  - ForAll L, Rumbling(L) <=> Exists L', Adjacent(L,L') & Forklift(L')
  - ForAll L, Safe(L) <=> ~Damaged(L) & ~Forklift(L)
"""
from collections import deque
from z3 import (
    Or, And, Not, Solver, unsat,
    DeclareSort, Function, BoolSort, Const, ForAll, Exists, Distinct,
)
from enum import Enum
# Use the warehouse environment defined earlier in this notebook.
HazardousWarehouseEnv = WarehouseEnv

class Direction(Enum):
    NORTH = (0, 1)
    EAST = (1, 0)
    SOUTH = (0, -1)
    WEST = (-1, 0)

    def delta(self):
        return self.value

    def turn_left(self):
        order = [Direction.NORTH, Direction.WEST, Direction.SOUTH, Direction.EAST]
        return order[(order.index(self) + 1) % 4]

    def turn_right(self):
        order = [Direction.NORTH, Direction.EAST, Direction.SOUTH, Direction.WEST]
        return order[(order.index(self) + 1) % 4]

class Action(Enum):
    FORWARD = 1
    TURN_LEFT = 2
    TURN_RIGHT = 3
    GRAB = 4
    EXIT = 5
# ---------------------------------------------------------------------------
# Z3 Entailment Check
# ---------------------------------------------------------------------------
def z3_entails(solver, query):
    """Check whether the solver's current assertions entail *query*.
    Uses the refutation method: push a checkpoint, assert Not(query),
    and check satisfiability.  If unsat, the negated query is
    inconsistent with the KB --- meaning the KB entails the query.
    Pop restores the solver to its previous state.
    """
    solver.push()
    solver.add(Not(query))
    result = solver.check() == unsat
    solver.pop()
    return result
# ---------------------------------------------------------------------------
# Adjacency
# ---------------------------------------------------------------------------
def get_adjacent(x, y, width=4, height=4):
    """Return the list of (x, y) positions adjacent to (x, y)."""
    result = []
    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nx, ny = x + dx, y + dy
        if 1 <= nx <= width and 1 <= ny <= height:
            result.append((nx, ny))
    return result
# ---------------------------------------------------------------------------
# Knowledge-Base Construction (Quantified FOL Encoding)
# ---------------------------------------------------------------------------
def build_warehouse_kb_fol(width=4, height=4):
    """Build a Z3 Solver using quantified first-order logic.
    The physics rules are expressed as single quantified sentences ---
    one per rule, independent of the grid size:
        ForAll L, Creaking(L) == Exists L', Adjacent(L,L') & Damaged(L')
        ForAll L, Rumbling(L) == Exists L', Adjacent(L,L') & Forklift(L')
        ForAll L, Safe(L) == And(Not(Damaged(L)), Not(Forklift(L)))
    Structural facts (adjacency, domain closure) require enumeration over
    grid squares, but these encode the grid topology, not the physics.
    Returns (solver, loc, predicates) where:
      - solver: Z3 Solver with all constraints
      - loc: dict mapping (x,y) to Z3 Location constants
      - predicates: dict mapping names to Z3 Function objects
    """
    Location = DeclareSort('Location')
    # Uninterpreted functions (predicates)
    Damaged_fn = Function('Damaged', Location, BoolSort())
    Forklift_fn = Function('Forklift', Location, BoolSort())
    Creaking_fn = Function('Creaking', Location, BoolSort())
    Rumbling_fn = Function('Rumbling', Location, BoolSort())
    Safe_fn = Function('Safe', Location, BoolSort())
    Adjacent_fn = Function('Adjacent', Location, Location, BoolSort())
    # Location constants --- one per grid square
    loc = {}
    for x in range(1, width + 1):
        for y in range(1, height + 1):
            loc[(x, y)] = Const(f'L_{x}_{y}', Location)
    solver = Solver()
    # --- Domain closure: every Location is one of our grid constants ---
    # Without this, ForAll could range over phantom locations that absorb
    # damage/forklift blame, breaking process-of-elimination reasoning.
    L = Const('L', Location)
    solver.add(ForAll(L,
        Or([L == loc[(x, y)]
            for x in range(1, width + 1)
            for y in range(1, height + 1)])
    ))
    # All location constants are distinct
    solver.add(Distinct(list(loc.values())))
    # --- Adjacency facts (closed-world) ---
    # Every pair of grid squares is either adjacent or not.
    for x in range(1, width + 1):
        for y in range(1, height + 1):
            adj_set = set(get_adjacent(x, y, width, height))
            for x2 in range(1, width + 1):
                for y2 in range(1, height + 1):
                    if (x2, y2) in adj_set:
                        solver.add(Adjacent_fn(loc[(x, y)], loc[(x2, y2)]))
                    else:
                        solver.add(Not(Adjacent_fn(loc[(x, y)], loc[(x2, y2)])))
    # --- Quantified physics rules ---
    # One sentence each --- no loop over grid squares.
    Lp = Const('Lp', Location)
    # Creaking rule
    solver.add(ForAll(L,
        Creaking_fn(L) == Exists(Lp, And(Adjacent_fn(L, Lp), Damaged_fn(Lp)))
    ))
    # Rumbling rule
    solver.add(ForAll(L,
        Rumbling_fn(L) == Exists(Lp, And(Adjacent_fn(L, Lp), Forklift_fn(Lp)))
    ))
    # Safety rule
    solver.add(ForAll(L,
        Safe_fn(L) == And(Not(Damaged_fn(L)), Not(Forklift_fn(L)))
    ))
    # --- Initial knowledge ---
    solver.add(Safe_fn(loc[(1, 1)]))
    predicates = {
        'Creaking': Creaking_fn,
        'Rumbling': Rumbling_fn,
        'Safe': Safe_fn,
        'Damaged': Damaged_fn,
        'Forklift': Forklift_fn,
        'Adjacent': Adjacent_fn,
    }
    return solver, loc, predicates
# ---------------------------------------------------------------------------
# Turning Helpers
# ---------------------------------------------------------------------------
_DIRECTION_ORDER = [Direction.NORTH, Direction.EAST, Direction.SOUTH, Direction.WEST]
def _direction_index(d):
    return _DIRECTION_ORDER.index(d)
def turns_between(current, target):
    """Return a list of TURN_LEFT / TURN_RIGHT actions to face *target*.
    Chooses the shortest rotation direction.
    """
    if current == target:
        return []
    ci = _direction_index(current)
    ti = _direction_index(target)
    right_steps = (ti - ci) % 4   # clockwise
    left_steps = (ci - ti) % 4    # counter-clockwise
    if right_steps <= left_steps:
        return [Action.TURN_RIGHT] * right_steps
    else:
        return [Action.TURN_LEFT] * left_steps
def delta_to_direction(dx, dy):
    """Map a movement delta to the Direction enum."""
    return {
        (0, 1): Direction.NORTH,
        (0, -1): Direction.SOUTH,
        (1, 0): Direction.EAST,
        (-1, 0): Direction.WEST,
    }[(dx, dy)]
# ---------------------------------------------------------------------------
# Z3 FOL Knowledge-Based Agent
# ---------------------------------------------------------------------------
class WarehouseZ3Agent:
    """A knowledge-based agent using Z3 FOL for the Hazardous Warehouse.
    Mirrors WarehouseKBAgent from warehouse_kb_agent.py with the same
    decision strategy, path planning, and action conversion logic.
    The difference is the reasoning engine: Z3 with quantified FOL
    rules replaces the DPLL-based propositional WarehouseKB.
    Decision strategy (in priority order):
      1. If the beacon is detected, GRAB the package.
      2. If carrying the package, navigate to (1,1) and EXIT.
      3. Otherwise, explore the nearest safe unvisited square.
      4. If no safe unvisited square is reachable, return to (1,1) and EXIT.
    """
    def __init__(self, env):
        self.env = env
        self.solver, self.loc, self.preds = build_warehouse_kb_fol(
            env.width, env.height
        )
        self.x = 1
        self.y = 1
        self.direction = Direction.EAST
        self.has_package = False
        self.visited = {(1, 1)}
        self.known_safe = {(1, 1)}
        self.known_dangerous = set()
        self.action_queue = []
        self.step_count = 0
    # ----- Percepts ----------------------------------------------------------
    def tell_percepts(self, percept):
        """Translate a Percept into Z3 FOL assertions and add to the solver."""
        L = self.loc[(self.x, self.y)]
        if percept.creaking:
            self.solver.add(self.preds['Creaking'](L))
        else:
            self.solver.add(Not(self.preds['Creaking'](L)))
        if percept.rumbling:
            self.solver.add(self.preds['Rumbling'](L))
        else:
            self.solver.add(Not(self.preds['Rumbling'](L)))
    # ----- Safety queries ----------------------------------------------------
    def update_safety(self):
        """Check entailment for every square whose status is still unknown."""
        for x in range(1, self.env.width + 1):
            for y in range(1, self.env.height + 1):
                pos = (x, y)
                if pos in self.known_safe or pos in self.known_dangerous:
                    continue
                L = self.loc[pos]
                if z3_entails(self.solver, self.preds['Safe'](L)):
                    self.known_safe.add(pos)
                elif z3_entails(self.solver, Not(self.preds['Safe'](L))):
                    self.known_dangerous.add(pos)
    # ----- Path planning -----------------------------------------------------
    def plan_path(self, start, goal_set):
        """BFS through known-safe squares from *start* to any cell in *goal_set*.
        Returns a list of (x, y) positions forming the path (including
        *start* and the reached goal), or None if no path exists.
        """
        queue = deque([(start, [start])])
        seen = {start}
        while queue:
            (cx, cy), path = queue.popleft()
            if (cx, cy) in goal_set:
                return path
            for nx, ny in get_adjacent(cx, cy, self.env.width, self.env.height):
                if (nx, ny) not in seen and (nx, ny) in self.known_safe:
                    seen.add((nx, ny))
                    queue.append(((nx, ny), path + [(nx, ny)]))
        return None
    def path_to_actions(self, path):
        """Convert a position path into a sequence of Actions.
        Returns (actions, final_direction) where *actions* is the list of
        TURN_LEFT / TURN_RIGHT / FORWARD actions and *final_direction* is
        the direction the robot faces after executing them all.
        """
        actions = []
        direction = self.direction
        for i in range(1, len(path)):
            dx = path[i][0] - path[i - 1][0]
            dy = path[i][1] - path[i - 1][1]
            target_dir = delta_to_direction(dx, dy)
            actions.extend(turns_between(direction, target_dir))
            actions.append(Action.FORWARD)
            direction = target_dir
        return actions, direction
    # ----- Decision logic ----------------------------------------------------
    def choose_action(self, percept):
        """Select the next action based on the current state of knowledge."""
        # Execute queued actions first (from a multi-step plan).
        if self.action_queue:
            return self.action_queue.pop(0)
        # 1. If the beacon is on, grab the package.
        if percept.beacon and not self.has_package:
            return Action.GRAB
        # 2. If carrying the package, navigate home and exit.
        if self.has_package:
            if (self.x, self.y) == (1, 1):
                return Action.EXIT
            path = self.plan_path((self.x, self.y), {(1, 1)})
            if path and len(path) > 1:
                actions, _ = self.path_to_actions(path)
                self.action_queue = actions[1:]
                return actions[0]
            # Already at (1,1) or can't find path — just exit.
            return Action.EXIT
        # 3. Explore the nearest safe unvisited square.
        safe_unvisited = self.known_safe - self.visited
        if safe_unvisited:
            path = self.plan_path((self.x, self.y), safe_unvisited)
            if path and len(path) > 1:
                actions, _ = self.path_to_actions(path)
                self.action_queue = actions[1:]
                return actions[0]
        # 4. Nothing left to explore — go home and exit.
        if (self.x, self.y) == (1, 1):
            return Action.EXIT
        path = self.plan_path((self.x, self.y), {(1, 1)})
        if path and len(path) > 1:
            actions, _ = self.path_to_actions(path)
            self.action_queue = actions[1:]
            self.action_queue.append(Action.EXIT)
            return actions[0]
        return Action.EXIT
    # ----- Execution ---------------------------------------------------------
    def execute_action(self, action):
        """Send *action* to the environment and update internal bookkeeping."""
        percept, reward, done, info = self.env.step(action)
        if action == Action.FORWARD and not percept.bump:
            dx, dy = self.direction.delta()
            self.x += dx
            self.y += dy
            self.visited.add((self.x, self.y))
        elif action == Action.TURN_LEFT:
            self.direction = self.direction.turn_left()
        elif action == Action.TURN_RIGHT:
            self.direction = self.direction.turn_right()
        elif action == Action.GRAB and info.get("grabbed"):
            self.has_package = True
        self.step_count += 1
        return percept, reward, done, info
    # ----- Main loop ---------------------------------------------------------
    def run(self, verbose=True):
        """Run the full perceive-tell-ask-act loop until the episode ends."""
        # Process the initial percept at (1, 1).
        percept = self.env._last_percept
        self.tell_percepts(percept)
        self.update_safety()
        if verbose:
            print(f"Start at ({self.x},{self.y}) facing {self.direction.name}")
            print(f"  Percept: {percept}")
            print(f"  Known safe: {sorted(self.known_safe)}")
        while True:
            action = self.choose_action(percept)
            percept, reward, done, info = self.execute_action(action)
            if verbose:
                print(f"\nStep {self.step_count}: {action.name}")
                print(f"  Position: ({self.x},{self.y}), Facing: {self.direction.name}")
                print(f"  Percept: {percept}")
                print(f"  Info: {info}")
            if done:
                if verbose:
                    print(f"\n{'=' * 40}")
                    print(f"Episode ended.  Reward: {self.env.total_reward:.0f}")
                    print(f"Steps taken: {self.step_count}")
                    success = info.get("exit") == "success"
                    print(f"Success: {success}")
                return
            # After moving to a new square, tell percepts and re-query safety.
            if action == Action.FORWARD and not percept.bump:
                self.tell_percepts(percept)
                self.update_safety()
                if verbose:
                    print(f"  Known safe: {sorted(self.known_safe)}")
                    print(f"  Known dangerous: {sorted(self.known_dangerous)}")
# ---------------------------------------------------------------------------
# Main — run on the example layout from the textbook
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    print("FOL agent code loaded.")
    print("Use the WarehouseEnv defined earlier in this notebook for experiments.")

FOL agent code loaded.
Use the WarehouseEnv defined earlier in this notebook for experiments.


# 3.7.4 Running the Agent
To run the Z3 FOL agent on the example layout from Section 3.2: The Hazardous Warehouse Environment:

In [ ]:
from types import SimpleNamespace

class HazardousAdapter:
    def __init__(self, base_env):
        self.base = base_env
        self.width = base_env.width
        self.height = base_env.height
        self.total_reward = 0.0
        self._heading = Direction.EAST
        self._last_percept = self._make_percept(bump=False)

    def _make_percept(self, bump=False):
        r, c = self.base.state.robot_pos
        tile = self.base.grid[r][c]
        beacon = (tile == "P" and not self.base.state.has_item)
        return SimpleNamespace(
            creaking=False,
            rumbling=False,
            beacon=beacon,
            bump=bump,
            beep=False,
        )

    def step(self, action):
        info = {}
        done = False
        reward = 0.0
        bump = False

        if action == Action.TURN_LEFT:
            self._heading = self._heading.turn_left()
            _, reward, terminated, truncated, info = self.base.step("WAIT")
            done = terminated or truncated
        elif action == Action.TURN_RIGHT:
            self._heading = self._heading.turn_right()
            _, reward, terminated, truncated, info = self.base.step("WAIT")
            done = terminated or truncated
        elif action == Action.FORWARD:
            move_map = {
                Direction.NORTH: "N",
                Direction.EAST: "E",
                Direction.SOUTH: "S",
                Direction.WEST: "W",
            }
            before = self.base.state.robot_pos
            _, reward, terminated, truncated, info = self.base.step(move_map[self._heading])
            after = self.base.state.robot_pos
            bump = (before == after)
            done = terminated or truncated
        elif action == Action.GRAB:
            _, reward, terminated, truncated, info = self.base.step("PICK")
            if reward > 0:
                info["grabbed"] = True
            done = terminated or truncated
        elif action == Action.EXIT:
            r, c = self.base.state.robot_pos
            success = (r, c) == (1, 1) and self.base.state.has_item
            info["exit"] = "success" if success else "failed"
            done = True
        else:
            _, reward, terminated, truncated, info = self.base.step("WAIT")
            done = terminated or truncated

        self.total_reward += reward
        percept = self._make_percept(bump=bump)
        self._last_percept = percept
        return percept, reward, done, info

# Use a smaller hazardous layout so quantified Z3 reasoning finishes quickly.
small_grid = [
    "######",
    "#...D#",
    "#.##.#",
    "#P...#",
    "######",
]
base_env = HazardousWarehouseEnv(grid=small_grid, start_pos=(1, 1), max_steps=20)
env = HazardousAdapter(base_env)
print("True state (hidden from agent):")
print(base_env.render())
agent = WarehouseZ3Agent(env)
agent.solver.set("timeout", 20)
agent.run(verbose=False)
print("Run completed.")
print(f"Total reward: {env.total_reward:.2f}")

True state (hidden from agent):
######
#R..D#
#.##.#
#P...#
######
